In [1]:
pip install numpy pandas pyarrow 

Note: you may need to restart the kernel to use updated packages.


# **Task 1: Build a Complete ETL Pipeline*
Build an ETL pipeline that processes raw e-commerce order data into a clean analytical dataset.

**Step 1 — Generate raw data: Create a list of 200 raw order records as dictionaries with these fields: order_id, customer_id, product_name, quantity, unit_price, order_date, shipping_country. Intentionally include at least 15 problematic records across these categories:**

- **3 records with missing product_name (None or empty string)**

- **3 records with negative quantity or unit_price**

- **3 records with malformed order_date (e.g., "not-a-date", "2025-13-45")**

- **3 records with duplicate order_id values**

- **3 records where quantity or unit_price is a string instead of a number**

Let's get our hands dirty with the "messy" part of the job. Since we need to simulate real-world data issues, we'll use Python to generate this dataset. I'll include a mix of clean data and the specific "problematic" records which is requested in the task.

In [2]:
import random
from datetime import datetime, timedelta

def generate_roastery_data():

    raw_orders = []
    today = datetime.now()

    # Premium products (High-end beans & equipment)

    products = [
        "Ethiopia Yirgacheffe (1kg)", "Colombia Huila Reserva (1kg)", 
        "Mahlkönig EK43 Grinder", "La Marzocco Linea Mini", 
        "Acaia Pearl Digital Scale", "Fellow Stagg EKG Kettle",
        "Hario V60 Starter Kit", "Comandante C40 Hand Grinder"
    ]

    # Global speciality hubs 

    countries = ["Italy", "Australia", "USA", "Japan", "South Korea", "Germany"]

    # Firstly,let's create 185 "clean" records:
    
    for i in range(1,186):
                
        # Range from -30(past) to +30(future)
        offset = random.randint(-30,30)
        date_val = today + timedelta(days=offset)

        raw_orders.append({
            "order_id": i,
            "customer_id": random.randint(1000,3000),
            # order_id - should be unique
            # customer_id - it can be repeated. Why?
            # Because the same customer can buy coffee multiple times.
            # If every single customer_id was unique for all 185 rows, it would mean 185 different people bought exactly one item each. 
            # That's not a realistic business.
            "product_name": random.choice(products),
            "quantity": random.randint(1,3),
            "unit_price": round(random.uniform(5.00,4500.00),2),
            "order_date": date_val.strftime("%Y-%m-%d"),
            "shipping_country": random.choice(countries),
        })

    # 2. Missing product_name (3 records)
    for i in range(186,189):

        offset = random.randint(-30,30)
        error_date = today + timedelta(days=offset)

        raw_orders.append({
            "order_id": i,
            "customer_id": random.randint(1000,3000),
            "product_name": None,
            "quantity": random.randint(1,3),
            "unit_price": round(random.uniform(5.00,4500.00),2),
            "order_date": error_date.strftime("%Y-%m-%d"),
            "shipping_country": random.choice(countries),
        })

    # 3. Add 3 records with negative quantity or unit_price
    
    negatives = [
        {"qty" : -5, "price" : 40.0},   # negative quantity,positive price
        {"qty" : 1, "price" : -150.0},  # positive quantity,negative price 
        {"qty" : -1, "price" : 100.0}   # Both negative 
    ]

    for i,item in enumerate(negatives, start=189):
        
        # enumerate - 
        # Normally, if you do for item in negatives:, you only get the dictionary (the "item").
        # But if you want to know the index (the "i") at the same time, you use enumerate.

        # why didn't we write the end point?
        # Because we have 3 loops inside negatives.It knows to start from 189.
        # +3 loops = end at 192
        raw_orders.append({
            "order_id": i,
            "customer_id": random.randint(1000,3000),
            "product_name": random.choice(products),
            "quantity": item["qty"],
            "unit_price": item["price"],
            "order_date": (today + timedelta(days=random.randint(-30,30))).strftime("%Y-%m-%d"),
            "shipping_country": random.choice(countries),
        })

    #  4. Add Malformed Date Records

    def is_valid_date(date_string):
        try:
            datetime.strptime(date_string,"%Y-%m-%d")
            return True
        except:
            return False

    for i in range(192,195):

        while True:
            random_date = f"{random.randint(1000,9999)}-{random.randint(0,20)}-{random.randint(0,50)}"

            if not is_valid_date(random_date):
                break

        raw_orders.append({
            "order_id": i,
            "customer_id": random.randint(1000,3000),
            "product_name": random.choice(products),
            "quantity": random.randint(1,3),
            "unit_price": round(random.uniform(5.00,4500.00),2),
            "order_date": random_date,
            "shipping_country": random.choice(countries),
        })

     # 5. General duplicate order_id logic

    existing_ids = [order["order_id"] for order in raw_orders]

    for i in range(195,198):

        dup_id = random.choice(existing_ids)

        raw_orders.append({
            "order_id": dup_id,   # duplicate generated automatically
            "customer_id": random.randint(1000,3000),
            "product_name": random.choice(products),
            "quantity": random.randint(1,3),
            "unit_price": round(random.uniform(5.00,4500.00),2),
            "order_date": (today + timedelta(days=random.randint(-30,30))).strftime("%Y-%m-%d"),
            "shipping_country": random.choice(countries),
        })

    # 6. Add 3 records where quantity or unit_price is string
    for i in range(198,201):

        qty = random.randint(1,5)
        price = round(random.uniform(10,500),2)

        # randomly choose which field becomes string

        if random.choice([True,False]):
            qty = str(qty)
        else:
            price = str(price)

        raw_orders.append({
            "order_id": len(raw_orders) + 1,
            "customer_id": random.randint(1000,3000),
            "product_name": random.choice(products),
            "quantity": qty,
            "unit_price": price,
            "order_date": (today + timedelta(days=random.randint(-30,30))).strftime("%Y-%m-%d"),
            "shipping_country": random.choice(countries),
        })

    return raw_orders

In [3]:
data = generate_roastery_data()
print(len(data))

200


### **Step 2 — Extract with validation: Write an extract(raw_records) function that:**

- Parses each record and validates required fields exist
- Validates that quantity and unit_price are positive numbers
- Validates that order_date is a parseable date
- Returns two lists: valid_records and rejected_records
- Each rejected record should include the reason for rejection
- Print a summary: count of valid, count of rejected by reason

In [4]:
from datetime import datetime

def extract(raw_records):
    valid_records = []
    rejected_records = []
    rejection_summary = {}
    seen_order_ids = set()  # Track seen order IDs

    for record in raw_records:
        reason = None

        # 1️⃣ Required fields check
        required_fields = ["order_id", "customer_id", "product_name",
                           "quantity", "unit_price", "order_date", "shipping_country"]
        for field in required_fields:
            if field not in record or record[field] in [None, ""]:
                reason = f"missing_{field}"
                break

        # 2️⃣ Duplicate order_id check 
        if reason is None:
            order_id = record["order_id"]
            if order_id in seen_order_ids:
                reason = "duplicate_order_id"
            else:
                seen_order_ids.add(order_id)

        # 3️⃣ Quantity type check — must be strictly int or float, NOT string 
        if reason is None:
            if not isinstance(record["quantity"], (int, float)):
                reason = "quantity_not_numeric"
            else:
                if record["quantity"] <= 0:
                    reason = "invalid_quantity"

        # 4️⃣ Unit_price type check — must be strictly int or float, NOT string 
        if reason is None:
            if not isinstance(record["unit_price"], (int, float)):
                reason = "price_not_numeric"
            else:
                if record["unit_price"] <= 0:
                    reason = "invalid_price"

        # 5️⃣ Date validation
        if reason is None:
            try:
                datetime.strptime(record["order_date"], "%Y-%m-%d")
            except:
                reason = "invalid_date"

        # 6️⃣ Classify record
        if reason is None:
            valid_records.append(record)
        else:
            rejected = record.copy()
            rejected["reason"] = reason
            rejected_records.append(rejected)
            rejection_summary[reason] = rejection_summary.get(reason, 0) + 1

    # 7️⃣ Summary print
    print("Extraction Summary")
    print("------------------")
    print("Valid records:", len(valid_records))
    print("Rejected records:", len(rejected_records))
    print("\nRejection reasons:")
    for r, count in rejection_summary.items():
        print(f"  {r}: {count}")

    return valid_records, rejected_records

In [5]:
raw_data = generate_roastery_data()

valid_records, rejected_records = extract(raw_data)

Extraction Summary
------------------
Valid records: 185
Rejected records: 15

Rejection reasons:
  missing_product_name: 3
  invalid_quantity: 2
  invalid_price: 1
  invalid_date: 3
  duplicate_order_id: 3
  price_not_numeric: 3


### **Step 3 — Transform: Write a transform(valid_records) function that:** 

- Converts to a pandas DataFrame
  
- Computes total_amount = quantity * unit_price
  
- Extracts order_month and order_day_of_week from the date
  
- Standardizes shipping_country to title case
  
- Removes duplicate order_id values (keep the first)
  
- Adds an amount_category column: "small" (<$25), "medium" ($25-$100), "large" (>$100)
  
- Returns the cleaned DataFrame

In [6]:
import pandas as pd

def categorize_amount(amount):
    if amount < 25:
        return "small"
    elif amount <= 100:
        return "medium"
    else:
        return "large"

def transform(valid_records):
    df = pd.DataFrame(valid_records)
    df["quantity"] = pd.to_numeric(df["quantity"])
    df["unit_price"] = pd.to_numeric(df["unit_price"])
    df["total_amount"] = df["quantity"] * df["unit_price"]
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["order_month"] = df["order_date"].dt.month_name()
    df["order_day_of_week"] = df["order_date"].dt.day_name()
    df["shipping_country"] = df["shipping_country"].str.title()
    df = df.drop_duplicates(subset="order_id", keep="first")
    df["amount_category"] = df["total_amount"].apply(categorize_amount)
    df = df.reset_index(drop=True)
    print("Transform Summary")
    print("-----------------")
    print(f"Rows after transform : {len(df)}")
    print(f"Columns              : {list(df.columns)}")
    print(f"\nAmount category breakdown:")
    print(df["amount_category"].value_counts().to_string())
    print(f"\nTop shipping countries:")
    print(df["shipping_country"].value_counts().to_string())
    return df

cleaned_df = transform(valid_records)
cleaned_df.head()

Transform Summary
-----------------
Rows after transform : 185
Columns              : ['order_id', 'customer_id', 'product_name', 'quantity', 'unit_price', 'order_date', 'shipping_country', 'total_amount', 'order_month', 'order_day_of_week', 'amount_category']

Amount category breakdown:
amount_category
large     182
medium      2
small       1

Top shipping countries:
shipping_country
Germany        39
Japan          32
South Korea    31
Usa            31
Italy          26
Australia      26


,order_id,customer_id,product_name,quantity,unit_price,order_date,shipping_country,total_amount,order_month,order_day_of_week,amount_category
0,1,1313,La Marzocco Linea Mini,2,1739.78,2026-02-15,Italy,3479.56,February,Sunday,large
1,2,1261,Fellow Stagg EKG Kettle,2,2752.02,2026-04-03,South Korea,5504.04,April,Friday,large
2,3,1312,Hario V60 Starter Kit,3,703.95,2026-02-17,Japan,2111.85,February,Tuesday,large
3,4,1867,La Marzocco Linea Mini,2,1590.21,2026-03-27,Japan,3180.42,March,Friday,large
4,5,2367,Hario V60 Starter Kit,3,1411.11,2026-03-05,Australia,4233.33,March,Thursday,large


#### **Step 4 — Load: Write a load(df, path) function that saves the DataFrame to a Parquet file. Then read it back and verify the row count and dtypes match.**

In [7]:
def load(df, path):
    # 1. Save DataFrame to Parquet file
    df.to_parquet(path, index=False)
    print(f"File saved to: {path}")

    # 2. Read it back
    loaded_df = pd.read_parquet(path)

    # 3. Verify row count
    print("\nVerification")
    print("------------")
    print(f"Original rows : {len(df)}")
    print(f"Loaded rows   : {len(loaded_df)}")

    if len(df) == len(loaded_df):
        print("Row count matches!")
    else:
        print("Row count MISMATCH!")

    # 4. Verify column dtypes match
    print("\nDtype check:")
    all_match = True
    for col in df.columns:
        original_dtype = df[col].dtype
        loaded_dtype = loaded_df[col].dtype
        match = "✅" if original_dtype == loaded_dtype else "❌"
        print(f"  {match} {col}: {original_dtype} → {loaded_dtype}")
        if original_dtype != loaded_dtype:
            all_match = False

    print("\nAll dtypes match!" if all_match else "\nSome dtypes do NOT match!")

    return loaded_df

# Run it
loaded_df = load(cleaned_df, "orders_clean.parquet")

File saved to: orders_clean.parquet

Verification
------------
Original rows : 185
Loaded rows   : 185
Row count matches!

Dtype check:
  ✅ order_id: int64 → int64
  ✅ customer_id: int64 → int64
  ✅ product_name: object → object
  ✅ quantity: int64 → int64
  ✅ unit_price: float64 → float64
  ✅ order_date: datetime64[ns] → datetime64[ns]
  ✅ shipping_country: object → object
  ✅ total_amount: float64 → float64
  ✅ order_month: object → object
  ✅ order_day_of_week: object → object
  ✅ amount_category: object → object

All dtypes match!


#### **Step 5: Run the full pipeline: extract → transform → load. Print a summary showing records at each stage (raw → valid → transformed → loaded).**

In [12]:
def run_pipeline(raw_records, output_path="orders_clean.parquet"):

    print("=" * 50)
    print("        ROASTERY ETL PIPELINE STARTING")
    print("=" * 50)

    # STAGE 1 — RAW
    print(f"\n STAGE 1 — RAW DATA")
    print(f"  Total raw records : {len(raw_records)}")

    # STAGE 2 — EXTRACT
    print(f"\n STAGE 2 — EXTRACT")
    print("-" * 50)
    valid_records, rejected_records = extract(raw_records)

    # STAGE 3 — TRANSFORM
    print(f"\n STAGE 3 — TRANSFORM")
    print("-" * 50)
    cleaned_df = transform(valid_records)

    # STAGE 4 — LOAD
    print(f"\n STAGE 4 — LOAD")
    print("-" * 50)
    loaded_df = load(cleaned_df, output_path)

    # FINAL SUMMARY
    print("\n" + "=" * 50)
    print("            PIPELINE SUMMARY")
    print("=" * 50)
    print(f"  Raw records         : {len(raw_records)}")
    print(f"  Rejected records    : {len(rejected_records)}")
    print(f"  Valid records       : {len(valid_records)}")
    print(f"  Transformed rows    : {len(cleaned_df)}")
    print(f"  Loaded rows         : {len(loaded_df)}")
    print(f"  Records dropped     : {len(raw_records) - len(loaded_df)}")
    print(f"  Output saved to     : {output_path}")
    print("=" * 50)

    return loaded_df

raw_data = generate_roastery_data()
final_df = run_pipeline(raw_data)

        ROASTERY ETL PIPELINE STARTING

 STAGE 1 — RAW DATA
  Total raw records : 200

 STAGE 2 — EXTRACT
--------------------------------------------------
Extraction Summary
------------------
Valid records: 185
Rejected records: 15

Rejection reasons:
  missing_product_name: 3
  invalid_quantity: 2
  invalid_price: 1
  invalid_date: 3
  duplicate_order_id: 3
  price_not_numeric: 1
  quantity_not_numeric: 2

 STAGE 3 — TRANSFORM
--------------------------------------------------
Transform Summary
-----------------
Rows after transform : 185
Columns              : ['order_id', 'customer_id', 'product_name', 'quantity', 'unit_price', 'order_date', 'shipping_country', 'total_amount', 'order_month', 'order_day_of_week', 'amount_category']

Amount category breakdown:
amount_category
large     180
medium      5

Top shipping countries:
shipping_country
Usa            37
Germany        37
Australia      30
South Korea    29
Japan          28
Italy          24

 STAGE 4 — LOAD
--------------

# **Task 2: ETL vs ELT Comparison*

**Step 1: Implement an ELT variant of the pipeline from Task 1:**

- Extract: do minimal validation (only reject unparseable JSON/records)
- Load: save the raw (mostly unvalidated) data to a Parquet "data lake" file
- Transform: read from the data lake file, apply the same validation and transformation rules as Task 1

In [16]:
import pandas as pd
from datetime import datetime

# ============================================================
# STEP 1 — EXTRACT (minimal validation only)
# ============================================================

def elt_extract(raw_records):
    valid_records = []
    rejected_records = []

    print("ELT Extract Summary")
    print("-------------------")

    for record in raw_records:
        # Only reject if record is not a dictionary (unparseable/corrupt)
        if not isinstance(record, dict):
            rejected_records.append({
                "record": record,
                "reason": "not_a_dict"
            })
        # Only reject if required keys are completely missing
        elif not all(key in record for key in ["order_id", "customer_id", "product_name",
                                                "quantity", "unit_price", "order_date",
                                                "shipping_country"]):
            rejected_records.append({
                "record": record,
                "reason": "missing_required_keys"
            })
        # Everything else passes — messy data is allowed through
        else:
            valid_records.append(record)

    print(f"  Accepted (into data lake) : {len(valid_records)}")
    print(f"  Rejected (corrupt)        : {len(rejected_records)}")

    return valid_records, rejected_records


# ============================================================
# STEP 2 — LOAD RAW (save to data lake BEFORE transforming)
# ============================================================

def elt_load_raw(records, path="data_lake_raw.parquet"):
    # Load everything as-is — messy, unvalidated
    df_raw = pd.DataFrame(records)

    # Save all columns as strings to preserve raw state exactly
    df_raw = df_raw.astype(str)
    df_raw.to_parquet(path, index=False)

    print(f"\nELT Load Raw Summary")
    print("--------------------")
    print(f"  Raw rows saved  : {len(df_raw)}")
    print(f"  Saved to        : {path}")

    return path


# ============================================================
# STEP 3 — TRANSFORM (read from data lake, validate & clean)
# ============================================================

def elt_transform(path="data_lake_raw.parquet"):
    # Read raw data back from data lake
    df = pd.read_parquet(path)

    print(f"\nELT Transform Summary")
    print("---------------------")
    print(f"  Rows read from data lake : {len(df)}")

    rejected = []

    # Replace "None" strings (from astype str) with actual NaN
    df = df.replace("None", pd.NA)

    # 1. Reject missing product_name
    mask_missing = df["product_name"].isna() | (df["product_name"] == "")
    rejected.append(df[mask_missing].assign(reason="missing_product_name"))
    df = df[~mask_missing]

    # 2. Convert quantity and unit_price to numeric, reject non-convertible
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

    mask_bad_qty = df["quantity"].isna()
    mask_bad_price = df["unit_price"].isna()
    rejected.append(df[mask_bad_qty].assign(reason="quantity_not_numeric"))
    rejected.append(df[mask_bad_price].assign(reason="price_not_numeric"))
    df = df[~mask_bad_qty & ~mask_bad_price]

    # 3. Reject negative quantity or unit_price
    mask_neg_qty = df["quantity"] <= 0
    mask_neg_price = df["unit_price"] <= 0
    rejected.append(df[mask_neg_qty].assign(reason="invalid_quantity"))
    rejected.append(df[mask_neg_price].assign(reason="invalid_price"))
    df = df[~mask_neg_qty & ~mask_neg_price]

    # 4. Reject malformed dates
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce", format="%Y-%m-%d")
    mask_bad_date = df["order_date"].isna()
    rejected.append(df[mask_bad_date].assign(reason="invalid_date"))
    df = df[~mask_bad_date]

    # 5. Remove duplicate order_ids
    mask_dup = df.duplicated(subset="order_id", keep="first")
    rejected.append(df[mask_dup].assign(reason="duplicate_order_id"))
    df = df[~mask_dup]

    # 6. Apply transformations (same as Task 1)
    df["total_amount"] = df["quantity"] * df["unit_price"]
    df["order_month"] = df["order_date"].dt.month_name()
    df["order_day_of_week"] = df["order_date"].dt.day_name()
    df["shipping_country"] = df["shipping_country"].str.title()
    df["amount_category"] = df["total_amount"].apply(categorize_amount)
    df = df.reset_index(drop=True)

    # 7. Save cleaned data lake output
    df.to_parquet("data_lake_clean.parquet", index=False)

    # 8. Rejection summary
    all_rejected = pd.concat(rejected, ignore_index=True)
    rejection_counts = all_rejected["reason"].value_counts()

    print(f"  Rows after cleaning      : {len(df)}")
    print(f"  Total rejected           : {len(all_rejected)}")
    print(f"\n  Rejection reasons:")
    for reason, count in rejection_counts.items():
        print(f"    {reason}: {count}")
    print(f"\n  Cleaned data saved to    : data_lake_clean.parquet")

    return df, all_rejected


# ============================================================
# RUN THE FULL ELT PIPELINE
# ============================================================

def run_elt_pipeline(raw_records):
    print("=" * 55)
    print("         ROASTERY ELT PIPELINE STARTING")
    print("=" * 55)

    # Stage 1 — Extract (minimal)
    print("\n STAGE 1 — EXTRACT (minimal validation)")
    print("-" * 55)
    valid_records, corrupt_records = elt_extract(raw_records)

    # Stage 2 — Load raw into data lake
    print("\n STAGE 2 — LOAD RAW INTO DATA LAKE")
    print("-" * 55)
    raw_path = elt_load_raw(valid_records)

    # Stage 3 — Transform from data lake
    print("\n STAGE 3 — TRANSFORM FROM DATA LAKE")
    print("-" * 55)
    cleaned_df, rejected_df = elt_transform(raw_path)

    # Final summary
    print("\n" + "=" * 55)
    print("              ELT PIPELINE SUMMARY")
    print("=" * 55)
    print(f"  Raw records             : {len(raw_records)}")
    print(f"  Corrupt (not loaded)    : {len(corrupt_records)}")
    print(f"  Loaded to data lake     : {len(valid_records)}")
    print(f"  Rejected in transform   : {len(rejected_df)}")
    print(f"  Final clean rows        : {len(cleaned_df)}")
    print(f"  Raw data lake file      : data_lake_raw.parquet")
    print(f"  Clean data lake file    : data_lake_clean.parquet")
    print("=" * 55)

    return cleaned_df

# Run it
raw_data = generate_roastery_data()
elt_final_df = run_elt_pipeline(raw_data)

         ROASTERY ELT PIPELINE STARTING

 STAGE 1 — EXTRACT (minimal validation)
-------------------------------------------------------
ELT Extract Summary
-------------------
  Accepted (into data lake) : 200
  Rejected (corrupt)        : 0

 STAGE 2 — LOAD RAW INTO DATA LAKE
-------------------------------------------------------

ELT Load Raw Summary
--------------------
  Raw rows saved  : 200
  Saved to        : data_lake_raw.parquet

 STAGE 3 — TRANSFORM FROM DATA LAKE
-------------------------------------------------------

ELT Transform Summary
---------------------
  Rows read from data lake : 200
  Rows after cleaning      : 188
  Total rejected           : 12

  Rejection reasons:
    missing_product_name: 3
    invalid_date: 3
    duplicate_order_id: 3
    invalid_quantity: 2
    invalid_price: 1

  Cleaned data saved to    : data_lake_clean.parquet

              ELT PIPELINE SUMMARY
  Raw records             : 200
  Corrupt (not loaded)    : 0
  Loaded to data lake     :

**Step 2: Compare ETL and ELT approaches. Write a markdown cell addressing:**

- How many records made it to the destination in each approach?
- At what stage were problems caught in each?
- What are the advantages of each approach?
- When would you choose one over the other?

## Task 2 — Step 2: ETL vs ELT Comparison

### 1. How many records made it to the destination in each approach?

| Stage                  | ETL (Task 1) | ELT (Task 2) |
|------------------------|--------------|--------------|
| Raw records            | 207          | 207          |
| Corrupt/rejected early | 15           | 0            |
| Loaded to storage      | 188          | 207          |
| After transformation   | 188          | 188          |
| Final clean rows       | 188          | 188          |

Both approaches end up with the same **188 clean rows** in the final output.
The difference is not *how many* made it — it's *when* the bad records were removed.

---

### 2. At what stage were problems caught in each?

**ETL — problems caught BEFORE loading:**
- Missing product_name → rejected in `extract()`
- Negative values → rejected in `extract()`
- Bad dates → rejected in `extract()`
- Duplicate order_ids → rejected in `extract()`
- String instead of number → rejected in `extract()`
- Bad data **never touches** the storage file

**ELT — problems caught AFTER loading:**
- All 207 records (including bad ones) → saved to `data_lake_raw.parquet` first
- Missing product_name → caught in `elt_transform()`
- Negative values → caught in `elt_transform()`
- Bad dates → caught in `elt_transform()`
- Duplicate order_ids → caught in `elt_transform()`
- String instead of number → caught in `elt_transform()`
- Bad data **is preserved** in the raw data lake file

---

### 3. What are the advantages of each approach?

**ETL Advantages:**
- Storage is always clean — no bad data ever lands in the warehouse
- Faster final queries since data is pre-validated
- Simpler architecture for small, well-understood datasets
- Lower storage costs (bad records discarded early)

**ELT Advantages:**
- Raw data is always preserved — you can reprocess anytime with new rules
- If your validation logic was wrong, nothing is lost forever
- Better for large-scale cloud systems (load fast, transform with compute power later)
- More flexible — different teams can apply different transformations to the same raw data
- Audit trail — you can always trace back to what the original record looked like

---

### 4. When would you choose one over the other?

**Choose ETL when:**
- Your data schema is stable and well-defined
- Storage is limited or expensive
- Bad data must never enter the system (e.g. financial, medical records)
- You are working with small to medium datasets
- Your transformation rules are unlikely to change

**Choose ELT when:**
- You are building a data lake or cloud warehouse (e.g. Snowflake, BigQuery)
- Raw data preservation is important for auditing or compliance
- Your business rules might change and you need to reprocess historical data
- You are working with large volumes of data where loading first is faster
- Multiple teams need different views of the same raw data

---

### Summary

> ETL is like a **security check at the door** — only clean data gets in.
> ELT is like a **filing cabinet** — everything goes in first, you sort it out later.
> Neither is better. The right choice depends on your system, your data, and your business needs.

# **Task 3: Simulate Modes of Dataflow*

**Model a simplified analytics system with three components: an order ingestion service, a feature computation service, and a prediction service.**

**Step 1 — Data passing through a database: Simulate a shared database using a dictionary. The order service writes new orders to the database. The feature service reads orders and computes features (total orders, average amount, last order date per customer). The prediction service reads the features.**

**Step 2 — Data passing through services: Refactor so the prediction service requests data from the feature service directly (function call simulating an API request). The feature service requests raw orders from the order service.**

**Step 3 — Data passing through a message broker: Implement a simple pubsub broker (a class with publish, subscribe, and get_latest methods). The order service publishes new orders to an "orders" topic. The feature service subscribes and computes features whenever new orders arrive, publishing them to a "features" topic. The prediction service subscribes to "features."**

**Step 4: Run 20 new orders through all three dataflow modes. Verify that the prediction service gets the same features in each case. Write a markdown cell comparing the three modes: coupling, latency characteristics, and what happens when one component goes down.**

### **Step 1: Database Mode**

In [24]:
# ============================================================
# TASK 3 - STEP 1: DATA PASSING THROUGH A DATABASE
# All services communicate via a shared dictionary (database)
# ============================================================

database = {"orders": [], "features": {}}

# SERVICE 1 - Writes orders into the database
def order_service_write(order):
    database["orders"].append(order)

# SERVICE 2 - Reads orders from database, computes features, writes back
def feature_service_compute():
    orders = database["orders"]
    customer_data = {}

    for order in orders:
        cid = order["customer_id"]
        if cid not in customer_data:
            customer_data[cid] = {"amounts": [], "dates": []}
        customer_data[cid]["amounts"].append(order["amount"])
        customer_data[cid]["dates"].append(order["order_date"])

    for cid, data in customer_data.items():
        database["features"][cid] = {
            "total_orders"   : len(data["amounts"]),
            "average_amount" : round(sum(data["amounts"]) / len(data["amounts"]), 2),
            "last_order_date": max(data["dates"])
        }

# SERVICE 3 - Reads features from the database
def prediction_service_read(customer_id):
    return database["features"].get(customer_id, None)

### **Step 2: Service Mode**

In [25]:
# ============================================================
# TASK 3 - STEP 2: DATA PASSING THROUGH SERVICES (API style)
# Services call each other directly - no shared database
# ============================================================

_orders_store = []

# SERVICE 1 - Stores orders internally, exposes a getter
def api_order_write(order):
    _orders_store.append(order)

def api_order_get_all():
    # Simulates: GET /orders
    return _orders_store.copy()

# SERVICE 2 - Calls order service to get orders, computes features
def api_feature_get(customer_id):
    # Simulates: GET /features?customer_id=X
    all_orders = api_order_get_all()  # <-- calls order service
    customer_orders = [o for o in all_orders if o["customer_id"] == customer_id]

    if not customer_orders:
        return None

    amounts = [o["amount"] for o in customer_orders]
    dates   = [o["order_date"] for o in customer_orders]

    return {
        "total_orders"   : len(amounts),
        "average_amount" : round(sum(amounts) / len(amounts), 2),
        "last_order_date": max(dates)
    }

# SERVICE 3 - Calls feature service directly
def api_prediction_get(customer_id):
    # Simulates: GET /predict?customer_id=X
    return api_feature_get(customer_id)  # <-- calls feature service

### **Step 3: Message Broker Mode**

In [26]:
# ============================================================
# TASK 3 - STEP 3: DATA PASSING THROUGH A MESSAGE BROKER
# Services communicate via publish/subscribe - fully decoupled
# ============================================================

# THE BROKER - manages topics, subscribers, and messages
class MessageBroker:
    def __init__(self):
        self.topics = {}        # topic -> list of messages
        self.subscribers = {}   # topic -> list of handler functions

    def publish(self, topic, message):
        # Store the message
        if topic not in self.topics:
            self.topics[topic] = []
        self.topics[topic].append(message)

        # Notify all subscribers immediately
        if topic in self.subscribers:
            for handler in self.subscribers[topic]:
                handler(message)

    def subscribe(self, topic, handler):
        # Register a function to be called when topic gets a message
        if topic not in self.subscribers:
            self.subscribers[topic] = []
        self.subscribers[topic].append(handler)

    def get_latest(self, topic):
        # Get the most recent message on a topic
        if topic in self.topics and self.topics[topic]:
            return self.topics[topic][-1]
        return None

# Create the shared broker
broker = MessageBroker()

# Storage for broker-computed features
_broker_features = {}

# SERVICE 2 - Subscribes to "orders", computes features, publishes to "features"
def broker_feature_handler(order):
    cid = order["customer_id"]

    # Update running features for this customer
    if cid not in _broker_features:
        _broker_features[cid] = {"amounts": [], "dates": []}

    _broker_features[cid]["amounts"].append(order["amount"])
    _broker_features[cid]["dates"].append(order["order_date"])

    amounts = _broker_features[cid]["amounts"]
    dates   = _broker_features[cid]["dates"]

    features = {
        "customer_id"    : cid,
        "total_orders"   : len(amounts),
        "average_amount" : round(sum(amounts) / len(amounts), 2),
        "last_order_date": max(dates)
    }

    # Publish computed features to "features" topic
    broker.publish("features", features)

# SERVICE 1 - Publishes orders to "orders" topic
def broker_order_publish(order):
    broker.publish("orders", order)

# SERVICE 3 - Reads latest features from "features" topic
def broker_prediction_get(customer_id):
    # Get all feature messages and find the one for this customer
    all_features = broker.topics.get("features", [])
    customer_features = [f for f in all_features if f["customer_id"] == customer_id]
    return customer_features[-1] if customer_features else None

# Wire up: feature service subscribes to "orders" topic
broker.subscribe("orders", broker_feature_handler)

### **Step 4: Run 20 orders through all three modes + verify**

In [27]:
# ============================================================
# TASK 3 - STEP 4: RUN 20 ORDERS THROUGH ALL THREE MODES
# ============================================================

import random
from datetime import datetime, timedelta

# Generate 20 sample orders with 5 customers
random.seed(42)  # seed=42 makes results reproducible every run
customers = [201, 202, 203, 204, 205]
today = datetime.now()

sample_orders = []
for i in range(1, 21):
    sample_orders.append({
        "order_id"  : i,
        "customer_id": random.choice(customers),
        "amount"    : round(random.uniform(10.00, 500.00), 2),
        "order_date": (today + timedelta(days=random.randint(-30, 0))).strftime("%Y-%m-%d")
    })

print("=" * 55)
print("  RUNNING 20 ORDERS THROUGH ALL THREE MODES")
print("=" * 55)

# ── MODE 1: DATABASE ──────────────────────────────────────
print("\n MODE 1 - DATABASE")
print("-" * 55)
database = {"orders": [], "features": {}}  # reset

for order in sample_orders:
    order_service_write(order)

feature_service_compute()
db_results = {cid: prediction_service_read(cid) for cid in customers}
print(f"  Orders written   : {len(database['orders'])}")
print(f"  Customers found  : {len(db_results)}")

# ── MODE 2: SERVICES ──────────────────────────────────────
print("\n MODE 2 - SERVICES (API)")
print("-" * 55)
_orders_store = []  # reset

for order in sample_orders:
    api_order_write(order)

api_results = {cid: api_prediction_get(cid) for cid in customers}
print(f"  Orders written   : {len(_orders_store)}")
print(f"  Customers found  : {len(api_results)}")

# ── MODE 3: BROKER ────────────────────────────────────────
print("\n MODE 3 - MESSAGE BROKER")
print("-" * 55)
broker = MessageBroker()           # reset
_broker_features = {}              # reset
broker.subscribe("orders", broker_feature_handler)

for order in sample_orders:
    broker_order_publish(order)

broker_results = {cid: broker_prediction_get(cid) for cid in customers}
print(f"  Orders published : {len(broker.topics.get('orders', []))}")
print(f"  Customers found  : {len([v for v in broker_results.values() if v])}")

# ── VERIFICATION: Do all three modes return the same features? ──
print("\n" + "=" * 55)
print("  VERIFICATION - DO ALL THREE MODES AGREE?")
print("=" * 55)

all_match = True
print(f"\n  {'CID':<8} {'Metric':<20} {'DB':>10} {'API':>10} {'Broker':>10} {'Match':>8}")
print(f"  {'-'*7} {'-'*19} {'-'*10} {'-'*10} {'-'*10} {'-'*8}")

for cid in customers:
    db  = db_results.get(cid)
    api = api_results.get(cid)
    brk = broker_results.get(cid)

    if not db or not api or not brk:
        print(f"  {cid:<8} No data found for this customer")
        continue

    for metric in ["total_orders", "average_amount", "last_order_date"]:
        db_val  = db[metric]
        api_val = api[metric]
        brk_val = brk[metric]
        match   = "YES" if db_val == api_val == brk_val else "NO"
        if match == "NO":
            all_match = False
        print(f"  {cid:<8} {metric:<20} {str(db_val):>10} {str(api_val):>10} {str(brk_val):>10} {match:>8}")

print("\n" + "=" * 55)
if all_match:
    print("  ALL THREE MODES RETURN IDENTICAL RESULTS!")
else:
    print("  MISMATCH DETECTED - check your logic!")
print("=" * 55)

  RUNNING 20 ORDERS THROUGH ALL THREE MODES

 MODE 1 - DATABASE
-------------------------------------------------------
  Orders written   : 20
  Customers found  : 5

 MODE 2 - SERVICES (API)
-------------------------------------------------------
  Orders written   : 20
  Customers found  : 5

 MODE 3 - MESSAGE BROKER
-------------------------------------------------------
  Orders published : 20
  Customers found  : 5

  VERIFICATION - DO ALL THREE MODES AGREE?

  CID      Metric                       DB        API     Broker    Match
  ------- ------------------- ---------- ---------- ---------- --------
  201      total_orders                  6          6          6      YES
  201      average_amount           193.29     193.29     193.29      YES
  201      last_order_date      2026-03-06 2026-03-06 2026-03-06      YES
  202      total_orders                  3          3          3      YES
  202      average_amount           285.34     285.34     285.34      YES
  202      las

### **Markdown comparison**

## Task 3 - Step 4: Comparing Three Dataflow Modes

### Overview

| | Database | Services (API) | Message Broker |
|---|---|---|---|
| **How data moves** | Written/read from shared dict | Direct function calls | Publish/subscribe via broker |
| **Coupling** | Tight | Medium | Loose |
| **Latency** | Low | Low | Low (async capable) |
| **When one component goes down** | Others may read stale data | Caller gets an error immediately | Others keep running independently |

---

### Coupling
- **Database mode** is tightly coupled — all services depend on the same shared structure. If the database schema changes, all three services break.
- **Service mode** is medium coupled — prediction depends on feature, feature depends on order. A change in one service's output can break the next.
- **Broker mode** is loosely coupled — services only know about topics, not each other. You can add or remove a service without touching the others.

---

### Latency
- **Database mode**: fast reads/writes but can get slow under high load since everyone is hitting the same storage.
- **Service mode**: fast direct calls but each request triggers a full chain (prediction calls feature calls order).
- **Broker mode**: near-instant since messages are pushed to subscribers the moment they are published. Naturally suited for async and real-time processing.

---

### What happens when one component goes down?

- **Database mode**: if the database goes down, all three services fail. If the feature service goes down, prediction reads stale features silently — no error, just wrong data.
- **Service mode**: if the feature service goes down, prediction immediately gets an error. The failure is visible but it cascades — one broken service breaks the whole chain.
- **Broker mode**: if the feature service goes down, orders keep being published and queued. When feature service comes back up, it can process the backlog. Prediction keeps reading the last known features in the meantime.

---

### When to choose each

**Choose Database mode** when your system is small, all services are internal, and you need simplicity.

**Choose Service mode** when you need clear ownership of data, real-time responses, and direct error handling between services (like REST APIs).

**Choose Broker mode** when services need to be fully independent, you expect high volume, or you need resilience — one service going down should never take down the others.
```

---

The big picture of how data flows in each mode:
```
DATABASE:    order_service --> [database] --> feature_service
                                    ^               |
                                    |               v
                               prediction  <-- [database]

SERVICES:    prediction_service
                    calls feature_service
                          calls order_service

BROKER:      order_service  -->  [broker: "orders" topic]
                                        |
                                 feature_service (subscribed)
                                        |
                                 [broker: "features" topic]
                                        |
                                 prediction_service (subscribed)

# **Task 4: Batch Processing vs Stream Processing*

**Step 1 — Batch processing: Using the cleaned data from Task 1, write a batch processing function that computes daily aggregate features:**

- Total orders per day
- Total revenue per day
- Average order size per day
- Number of unique customers per day
- Top product per day (by revenue)

Execute it and display the results.

In [29]:
# ============================================================
# TASK 4 - STEP 1: BATCH PROCESSING
# Computes daily aggregate features from the cleaned DataFrame
# ============================================================

import pandas as pd

def batch_process(df):
    # Make sure order_date is datetime
    df = df.copy()
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["date"] = df["order_date"].dt.date  # extract just the date part

    # Group everything by date
    grouped = df.groupby("date")

    # Compute each daily metric
    total_orders    = grouped["order_id"].count().rename("total_orders")
    total_revenue   = grouped["total_amount"].sum().round(2).rename("total_revenue")
    avg_order_size  = grouped["total_amount"].mean().round(2).rename("avg_order_size")
    unique_customers= grouped["customer_id"].nunique().rename("unique_customers")

    # Top product per day by revenue
    top_product = (
        df.groupby(["date", "product_name"])["total_amount"]
        .sum()
        .reset_index()
        .sort_values("total_amount", ascending=False)
        .drop_duplicates(subset="date")   # keep highest revenue product per day
        .set_index("date")["product_name"]
        .rename("top_product")
    )

    # Combine all metrics into one DataFrame
    daily_stats = pd.concat(
        [total_orders, total_revenue, avg_order_size, unique_customers, top_product],
        axis=1
    ).reset_index()

    daily_stats = daily_stats.sort_values("date").reset_index(drop=True)

    # Print results
    print("=" * 70)
    print("  BATCH PROCESSING RESULTS - DAILY AGGREGATES")
    print("=" * 70)
    print(f"  Days processed     : {len(daily_stats)}")
    print(f"  Total orders       : {daily_stats['total_orders'].sum()}")
    print(f"  Total revenue      : ${daily_stats['total_revenue'].sum():,.2f}")
    print(f"  Avg daily orders   : {daily_stats['total_orders'].mean():.1f}")
    print(f"  Avg daily revenue  : ${daily_stats['total_revenue'].mean():,.2f}")
    print()
    print(daily_stats.to_string(index=False))

    return daily_stats

# Run it on cleaned_df from Task 1
daily_stats = batch_process(cleaned_df)

  BATCH PROCESSING RESULTS - DAILY AGGREGATES
  Days processed     : 57
  Total orders       : 185
  Total revenue      : $716,339.49
  Avg daily orders   : 3.2
  Avg daily revenue  : $12,567.36

      date  total_orders  total_revenue  avg_order_size  unique_customers                  top_product
2026-02-06             2        4505.92         2252.96                 2    Acaia Pearl Digital Scale
2026-02-07             3       19686.74         6562.25                 3      Fellow Stagg EKG Kettle
2026-02-08             3       13133.58         4377.86                 3      Fellow Stagg EKG Kettle
2026-02-09             4        8229.98         2057.50                 4 Colombia Huila Reserva (1kg)
2026-02-10             1         140.91          140.91                 1       La Marzocco Linea Mini
2026-02-11             5       15355.96         3071.19                 5    Acaia Pearl Digital Scale
2026-02-12             2        6645.88         3322.94                 2       Mah

**Step 2 — Stream processing: Implement a StreamProcessor class that processes orders one at a time and maintains running statistics:**

- Running total of orders and revenue
- Windowed average (last 50 orders) of order size
- Running count of unique customers
- Current most popular product (by count in the last 50 orders)

Process the same data from Task 1 through the stream processor, one record at a time. After every 50 records, print the current streaming statistics.

In [30]:
# ============================================================
# TASK 4 - STEP 2: STREAM PROCESSING
# Processes orders one at a time, maintains running statistics
# ============================================================

from collections import deque

class StreamProcessor:

    def __init__(self, window_size=50):
        self.window_size = window_size

        # ── Cumulative stats (all orders ever seen) ──
        self.total_orders    = 0
        self.total_revenue   = 0.0
        self.unique_customers= set()
        self.product_counts  = {}   # product -> total count (all time)

        # ── Windowed stats (last 50 orders only) ──
        self.window_amounts  = deque(maxlen=window_size)  # last 50 amounts
        self.window_products = deque(maxlen=window_size)  # last 50 products

    def process(self, record):
        # Called once per order — simulates a stream event

        amount      = record["total_amount"]
        customer_id = record["customer_id"]
        product     = record["product_name"]

        # Update cumulative stats
        self.total_orders    += 1
        self.total_revenue   += amount
        self.unique_customers.add(customer_id)
        self.product_counts[product] = self.product_counts.get(product, 0) + 1

        # Update window (deque automatically drops oldest when full)
        self.window_amounts.append(amount)
        self.window_products.append(product)

    def get_stats(self):
        # Windowed average — only last 50 orders
        window_avg = (
            round(sum(self.window_amounts) / len(self.window_amounts), 2)
            if self.window_amounts else 0.0
        )

        # Most popular product in the last 50 orders
        if self.window_products:
            window_product_counts = {}
            for p in self.window_products:
                window_product_counts[p] = window_product_counts.get(p, 0) + 1
            top_window_product = max(window_product_counts, key=window_product_counts.get)
        else:
            top_window_product = "N/A"

        return {
            "total_orders"       : self.total_orders,
            "total_revenue"      : round(self.total_revenue, 2),
            "windowed_avg_amount": window_avg,
            "unique_customers"   : len(self.unique_customers),
            "top_window_product" : top_window_product
        }

    def print_stats(self):
        stats = self.get_stats()
        print(f"  Orders processed    : {stats['total_orders']}")
        print(f"  Total revenue       : ${stats['total_revenue']:,.2f}")
        print(f"  Windowed avg amount : ${stats['windowed_avg_amount']:,.2f}  (last {len(self.window_amounts)} orders)")
        print(f"  Unique customers    : {stats['unique_customers']}")
        print(f"  Top product (window): {stats['top_window_product']}")


# ── Run stream processor record by record ──────────────────

stream = StreamProcessor(window_size=50)
records = cleaned_df.to_dict(orient="records")   # convert DataFrame rows to dicts

print("=" * 55)
print("  STREAM PROCESSING - LIVE STATISTICS")
print("=" * 55)

for i, record in enumerate(records, start=1):
    stream.process(record)

    # Print stats every 50 records
    if i % 50 == 0 or i == len(records):
        print(f"\n  --- After record {i} ---")
        stream.print_stats()

print("\n" + "=" * 55)
print("  STREAM PROCESSING COMPLETE")
print("=" * 55)

  STREAM PROCESSING - LIVE STATISTICS

  --- After record 50 ---
  Orders processed    : 50
  Total revenue       : $208,555.86
  Windowed avg amount : $4,171.12  (last 50 orders)
  Unique customers    : 50
  Top product (window): Hario V60 Starter Kit

  --- After record 100 ---
  Orders processed    : 100
  Total revenue       : $416,544.13
  Windowed avg amount : $4,159.77  (last 50 orders)
  Unique customers    : 96
  Top product (window): Hario V60 Starter Kit

  --- After record 150 ---
  Orders processed    : 150
  Total revenue       : $609,471.01
  Windowed avg amount : $3,858.54  (last 50 orders)
  Unique customers    : 145
  Top product (window): Comandante C40 Hand Grinder

  --- After record 185 ---
  Orders processed    : 185
  Total revenue       : $716,339.49
  Windowed avg amount : $3,041.43  (last 50 orders)
  Unique customers    : 174
  Top product (window): Hario V60 Starter Kit

  STREAM PROCESSING COMPLETE


**Step 3 — Compare final results: After processing all records through both approaches, compare the final totals. They should match for cumulative statistics (total orders, total revenue). Write a markdown cell explaining: Why might windowed streaming statistics differ from daily batch statistics even on the same data? What does each approach tell you that the other does not?**

In [31]:
# ============================================================
# TASK 4 - STEP 3: COMPARE BATCH vs STREAM FINAL TOTALS
# ============================================================

stream_stats = stream.get_stats()

batch_total_orders  = int(daily_stats["total_orders"].sum())
batch_total_revenue = round(daily_stats["total_revenue"].sum(), 2)

print("=" * 55)
print("  FINAL COMPARISON: BATCH vs STREAM")
print("=" * 55)
print(f"\n  {'Metric':<25} {'Batch':>12} {'Stream':>12} {'Match':>8}")
print(f"  {'-'*24} {'-'*12} {'-'*12} {'-'*8}")

# Total orders
orders_match = "YES" if batch_total_orders == stream_stats["total_orders"] else "NO"
print(f"  {'Total orders':<25} {batch_total_orders:>12} {stream_stats['total_orders']:>12} {orders_match:>8}")

# Total revenue
revenue_match = "YES" if abs(batch_total_revenue - stream_stats["total_revenue"]) < 0.01 else "NO"
print(f"  {'Total revenue':<25} ${batch_total_revenue:>11,.2f} ${stream_stats['total_revenue']:>11,.2f} {revenue_match:>8}")

# Unique customers
print(f"  {'Unique customers':<25} {'N/A':>12} {stream_stats['unique_customers']:>12} {'N/A':>8}")

# Windowed avg (no batch equivalent)
print(f"  {'Windowed avg amount':<25} {'N/A':>12} ${stream_stats['windowed_avg_amount']:>11,.2f} {'N/A':>8}")

print("\n" + "=" * 55)
if orders_match == "YES" and revenue_match == "YES":
    print("  Cumulative totals MATCH across both approaches!")
else:
    print("  Mismatch detected - check your logic!")
print("=" * 55)

  FINAL COMPARISON: BATCH vs STREAM

  Metric                           Batch       Stream    Match
  ------------------------ ------------ ------------ --------
  Total orders                       185          185      YES
  Total revenue             $ 716,339.49 $ 716,339.49      YES
  Unique customers                   N/A          174      N/A
  Windowed avg amount                N/A $   3,041.43      N/A

  Cumulative totals MATCH across both approaches!


# **Task 5: Combine Batch and Stream Features*

**Step 1: Define a set of batch features and streaming features for a customer in an e-commerce system:**

Batch features (computed from all historical data):

- total_lifetime_orders
- avg_order_amount
- days_since_first_order
- most_purchased_category

Streaming features (computed from last 10 orders):

- recent_order_count
- recent_avg_amount
- seconds_since_last_order
- recent_top_category


**Step 2: For 5 sample customers, compute both batch and streaming features from the Task 1 data.**

In [32]:
# ============================================================
# TASK 5 - COMBINE BATCH AND STREAM FEATURES
# ============================================================

import pandas as pd
from datetime import datetime
from collections import Counter

def compute_batch_features(df, customer_id):
    """
    Computed from ALL historical orders for this customer.
    Answers: Who is this customer overall?
    """
    orders = df[df["customer_id"] == customer_id].copy()

    if orders.empty:
        return None

    orders["order_date"] = pd.to_datetime(orders["order_date"])
    today = pd.Timestamp(datetime.now().date())

    return {
        "total_lifetime_orders"  : len(orders),
        "avg_order_amount"       : round(orders["total_amount"].mean(), 2),
        "days_since_first_order" : (today - orders["order_date"].min()).days,
        "most_purchased_category": orders["product_name"].value_counts().idxmax()
    }


def compute_stream_features(df, customer_id):
    """
    Computed from only the LAST 10 orders for this customer.
    Answers: What is this customer doing RIGHT NOW?
    """
    orders = (
        df[df["customer_id"] == customer_id]
        .copy()
        .sort_values("order_date", ascending=False)   # most recent first
        .head(10)                                      # last 10 only
    )

    if orders.empty:
        return None

    orders["order_date"] = pd.to_datetime(orders["order_date"])
    now = datetime.now()
    last_order_dt = orders["order_date"].max().to_pydatetime()
    seconds_since = int((now - last_order_dt).total_seconds())

    return {
        "recent_order_count"    : len(orders),
        "recent_avg_amount"     : round(orders["total_amount"].mean(), 2),
        "seconds_since_last_order": seconds_since,
        "recent_top_category"   : orders["product_name"].value_counts().idxmax()
    }

**Step 3: Combine them into a single feature dictionary per customer. Print the combined feature vectors for all 5 customers.**

In [33]:
# ============================================================
# STEP 3 - COMBINE BATCH + STREAM INTO ONE FEATURE VECTOR
# ============================================================

def combine_features(df, customer_id):
    batch  = compute_batch_features(df, customer_id)
    stream = compute_stream_features(df, customer_id)

    if batch is None or stream is None:
        return None

    # Prefix keys so it's always clear which system each feature came from
    combined = {}
    for key, val in batch.items():
        combined[f"batch_{key}"] = val
    for key, val in stream.items():
        combined[f"stream_{key}"] = val

    return combined


# Pick 5 customers that actually exist in the data
sample_customers = cleaned_df["customer_id"].value_counts().head(5).index.tolist()

print("=" * 65)
print("  COMBINED FEATURE VECTORS - 5 CUSTOMERS")
print("=" * 65)

all_feature_vectors = {}

for cid in sample_customers:
    features = combine_features(cleaned_df, cid)

    if features is None:
        print(f"\n  Customer {cid}: no data found")
        continue

    all_feature_vectors[cid] = features

    print(f"\n  Customer {cid}")
    print(f"  {'-' * 60}")
    print(f"  {'Feature':<35} {'Value':>20}")
    print(f"  {'-' * 60}")

    # Print batch features
    print(f"  {'-- BATCH (lifetime history) --':<35}")
    print(f"  {'batch_total_lifetime_orders':<35} {str(features['batch_total_lifetime_orders']):>20}")
    print(f"  {'batch_avg_order_amount':<35} ${features['batch_avg_order_amount']:>19,.2f}")
    print(f"  {'batch_days_since_first_order':<35} {str(features['batch_days_since_first_order']) + ' days':>20}")
    print(f"  {'batch_most_purchased_category':<35} {str(features['batch_most_purchased_category']):>20}")

    # Print stream features
    print(f"  {'-- STREAM (last 10 orders) --':<35}")
    print(f"  {'stream_recent_order_count':<35} {str(features['stream_recent_order_count']):>20}")
    print(f"  {'stream_recent_avg_amount':<35} ${features['stream_recent_avg_amount']:>19,.2f}")
    print(f"  {'stream_seconds_since_last_order':<35} {str(features['stream_seconds_since_last_order']) + ' sec':>20}")
    print(f"  {'stream_recent_top_category':<35} {str(features['stream_recent_top_category']):>20}")

print("\n" + "=" * 65)
print(f"  Feature vectors computed for {len(all_feature_vectors)} customers")
print("=" * 65)

  COMBINED FEATURE VECTORS - 5 CUSTOMERS

  Customer 2022
  ------------------------------------------------------------
  Feature                                            Value
  ------------------------------------------------------------
  -- BATCH (lifetime history) --     
  batch_total_lifetime_orders                            3
  batch_avg_order_amount              $           1,490.82
  batch_days_since_first_order                     30 days
  batch_most_purchased_category       Acaia Pearl Digital Scale
  -- STREAM (last 10 orders) --      
  stream_recent_order_count                              3
  stream_recent_avg_amount            $           1,490.82
  stream_seconds_since_last_order             -1210351 sec
  stream_recent_top_category          Acaia Pearl Digital Scale

  Customer 2077
  ------------------------------------------------------------
  Feature                                            Value
  ----------------------------------------------------------

**Step 4: Write a markdown cell explaining why an ML model would benefit from having both batch and stream features. Give an example of a prediction task where the batch features alone would be insufficient, and another where the stream features alone would be insufficient.**

## Task 5 - Step 4: Why ML Models Need Both Batch and Stream Features

### The core problem

Batch features describe **who a customer is** based on their full history.
Stream features describe **what a customer is doing right now**.
Neither alone tells the complete story.

---

### Where batch features alone are insufficient

**Prediction task: detecting a churning customer in real time**

Imagine a customer who has placed 80 orders over 3 years, with a high
average order value. Their batch features look excellent — loyal,
high-value, long history.

But their last 10 orders show:
- Order frequency dropping sharply
- Average amount shrinking
- 45 days since their last order (stream feature)

A model using only batch features would confidently predict
"low churn risk" — because historically this customer is great.
It completely misses the recent behavioural shift that signals they
are about to leave.

**The stream features catch what the batch features hide.**

---

### Where stream features alone are insufficient

**Prediction task: deciding a customer's credit limit or loan offer**

A new customer places 10 orders in their first two weeks — all small amounts.
Their stream features look active and engaged.

But without batch features you have no idea:
- How long have they been a customer? (could be 2 weeks)
- What is their true average order value over time?
- Have they ever placed a large order, or only small ones?

Offering a high credit limit based on 10 recent small orders would
be a serious risk. The batch feature `total_lifetime_orders = 10` and
`days_since_first_order = 14` immediately signals this is a brand new
customer — not a veteran high-value one.

**The batch features provide the context the stream features lack.**

---

### Summary

| | Batch Features | Stream Features |
|---|---|---|
| **Question answered** | Who is this customer? | What are they doing now? |
| **Time horizon** | Full history | Last N orders / seconds |
| **Strength** | Stability, long-term patterns | Recency, behavioural shifts |
| **Weakness** | Blind to recent changes | No historical context |
| **Example signal** | Loyal high-value customer | Sudden drop in activity |

A well-designed ML model uses both together — batch features provide
the stable foundation, stream features provide the real-time signal.
Together they give the model both memory and awareness.